In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent/'data'))

from make_data import *
import torch #For my text editor
import torch.nn as nn
import torch.nn.functional as F
device = torch.device('cuda')
from torch.utils.data import TensorDataset, DataLoader

In [2]:
train = ProteinDataset('../../../data/train.csv')
test = ProteinDataset('../../../data/test.csv')
X_train = train.tokens.float()
X_test = test.tokens.float()

In [22]:
class decoder_only_layer(nn.TransformerEncoderLayer):
    ## We will reuse code from nn.TransformerEncoderLayer
    def __init__(self, d_model, nhead, seq_len = 128, device=None):
        super().__init__(d_model=d_model, nhead=nhead, batch_first=True,norm_first = True)
        # We will hard code it to be causal, and assume all data is of shape d_model X seq_len. Since shorter proteins are padded with 0s

        self.register_buffer("mask",
            torch.triu(
                torch.ones(seq_len, seq_len, dtype=torch.bool),
                diagonal=1))
        
            
    def forward(self, protein, padding_mask):
        return super().forward(
            protein,
            src_mask = self.mask,
            src_key_padding_mask=padding_mask,
            is_causal = True
        )      

class Transformer_model(nn.Module):
    def __init__(self,d_model,nhead,d_ff,vocab_size=23,seq_len = 128, device=None):
        super().__init__()
        self.device=device
        self.seq_embedding = nn.Embedding(vocab_size, d_model, device=device)
        self.pos_embedding = nn.Embedding(seq_len, d_model,device=device)
        self.dol1=decoder_only_layer(d_model=d_model, nhead=nhead, seq_len=seq_len, device = device)
        self.dol2=decoder_only_layer(d_model=d_model, nhead=nhead, seq_len=seq_len, device = device)
        self.ff1 =nn.Linear(d_model,d_ff)
        self.relu=nn.ReLU()
        self.ff2 =nn.Linear(d_ff,23) # 20 amino acids + 1 EOS + BOS + PAD
        
    def forward(self, x):
        padding_mask = (x==0)
        positions = torch.arange(x.size(1), device=x.device)
        x=self.seq_embedding(x) + self.pos_embedding(positions)
        x=self.dol1(x, padding_mask)
        x=self.dol2(x, padding_mask)
        x=self.ff1(x)
        x=self.relu(x)
        x=self.ff2(x)
        
        return(x)

v1 = Transformer_model(d_model=128, nhead=4, d_ff=100,device=device).to(device)
v2 = Transformer_model(d_model=256, nhead=4, d_ff=100,device=device).to(device)
v4 = Transformer_model(d_model=128, nhead=4, d_ff=200,device=device).to(device)
v4 = Transformer_model(d_model=256, nhead=4, d_ff=200,device=device).to(device)


In [4]:
X_train = X_train.long()
X_test = X_test.long()
train_dataset = TensorDataset(X_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=32
)

test_dataset = TensorDataset(X_test)
test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [5]:
#Test forward pass
v1.eval()
for (batch,) in train_loader:
    batch = batch[:,:-1].to(device)
    output = v1(batch)
print(output.shape)

torch.Size([20, 128, 23])


In [ ]:
#Train v1
optim = torch.optim.Adam(v1.parameters(), lr=0.0001)
loss = nn.CrossEntropyLoss(ignore_index=0)

epochs = 20

for epoch in range(epochs):
    # Training
    v1.train()
    total_loss = 0

    for (batch,) in train_loader:
        optim.zero_grad()

        batch = batch.to(device)
        X = batch[:, :-1]
        Y = batch[:, 1:]

        output_train = v1(X)
        Loss = loss(output_train.reshape(-1, 23), Y.reshape(-1))

        total_loss += Loss.item()

        Loss.backward()
        optim.step()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    v1.eval()
    total_val_loss = 0
    id_true = 0 #for checking %correct aa assignments
    id_total = 0 

    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)

            X = batch[:, :-1]
            Y = batch[:, 1:]

            output_test = v1(X)
            predictions = output_test.argmax(dim=-1)
            id_mask = Y!=0
            id_true += ((predictions == Y)*id_mask).sum()
            id_total += id_mask.sum() 
            
            val_loss = loss(output_test.reshape(-1, 23), Y.reshape(-1))

            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(test_loader)

    if epoch%5==0:
        print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Validation Loss: {avg_val_loss:.4f} |"
            f"Validation %id: {id_true/id_total}")


Epoch 1/20 | Train Loss: 2.5233 | Validation Loss: 2.3089 |Validation %id: 0.31547975540161133
Epoch 6/20 | Train Loss: 1.8459 | Validation Loss: 1.9505 |Validation %id: 0.40313079953193665
Epoch 11/20 | Train Loss: 1.7067 | Validation Loss: 1.8679 |Validation %id: 0.4331842362880707
Epoch 16/20 | Train Loss: 1.6284 | Validation Loss: 1.8250 |Validation %id: 0.4473579227924347


In [ ]:
optim = torch.optim.Adam(v2.parameters(), lr=0.0001)
loss = nn.CrossEntropyLoss(ignore_index=0)

epochs = 20

for epoch in range(epochs):
    # Training
    v2.train()
    total_loss = 0

    for (batch,) in train_loader:
        optim.zero_grad()

        batch = batch.to(device)
        X = batch[:, :-1]
        Y = batch[:, 1:]

        output_train = v2(X)
        Loss = loss(output_train.reshape(-1, 23), Y.reshape(-1))

        total_loss += Loss.item()

        Loss.backward()
        optim.step()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    v2.eval()
    total_val_loss = 0
    id_true = 0 #for checking %correct aa assignments
    id_total = 0 

    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)

            X = batch[:, :-1]
            Y = batch[:, 1:]

            output_test = v2(X)
            predictions = output_test.argmax(dim=-1)
            id_mask = Y!=0
            id_true += ((predictions == Y)*id_mask).sum()
            id_total += id_mask.sum() 
            
            val_loss = loss(output_test.reshape(-1, 23), Y.reshape(-1))

            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(test_loader)

    if epoch%5==0:
        print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Validation Loss: {avg_val_loss:.4f} |"
            f"Validation %id: {id_true/id_total}")


Epoch 1/20 | Train Loss: 2.3806 | Validation Loss: 2.1574 |Validation %id: 0.35055696964263916
Epoch 6/20 | Train Loss: 1.6763 | Validation Loss: 1.8482 |Validation %id: 0.4345228672027588
Epoch 11/20 | Train Loss: 1.5345 | Validation Loss: 1.7713 |Validation %id: 0.4636261463165283
Epoch 16/20 | Train Loss: 1.4514 | Validation Loss: 1.7231 |Validation %id: 0.47973689436912537


In [23]:
optim = torch.optim.Adam(v3.parameters(), lr=0.0001)
loss = nn.CrossEntropyLoss(ignore_index=0)

epochs = 21

for epoch in range(epochs):
    # Training
    v3.train()
    total_loss = 0

    for (batch,) in train_loader:
        optim.zero_grad()

        batch = batch.to(device)
        X = batch[:, :-1]
        Y = batch[:, 1:]

        output_train = v3(X)
        Loss = loss(output_train.reshape(-1, 23), Y.reshape(-1))

        total_loss += Loss.item()

        Loss.backward()
        optim.step()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    v3.eval()
    total_val_loss = 0
    id_true = 0 #for checking %correct aa assignments
    id_total = 0 

    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)

            X = batch[:, :-1]
            Y = batch[:, 1:]

            output_test = v3(X)
            predictions = output_test.argmax(dim=-1)
            id_mask = Y!=0
            id_true += ((predictions == Y)*id_mask).sum()
            id_total += id_mask.sum() 
            
            val_loss = loss(output_test.reshape(-1, 23), Y.reshape(-1))

            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(test_loader)

    if epoch%5==0:
        print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Validation Loss: {avg_val_loss:.4f} |"
            f"Validation %id: {id_true/id_total}")


KeyboardInterrupt: 

In [24]:
optim = torch.optim.Adam(v4.parameters(), lr=0.0001)
loss = nn.CrossEntropyLoss(ignore_index=0)

epochs = 21

for epoch in range(epochs):
    # Training
    v4.train()
    total_loss = 0

    for (batch,) in train_loader:
        optim.zero_grad()

        batch = batch.to(device)
        X = batch[:, :-1]
        Y = batch[:, 1:]

        output_train = v4(X)
        Loss = loss(output_train.reshape(-1, 23), Y.reshape(-1))

        total_loss += Loss.item()

        Loss.backward()
        optim.step()

    avg_train_loss = total_loss / len(train_loader)

    # Validation
    v4.eval()
    total_val_loss = 0
    id_true = 0 #for checking %correct aa assignments
    id_total = 0 

    with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)

            X = batch[:, :-1]
            Y = batch[:, 1:]

            output_test = v4(X)
            predictions = output_test.argmax(dim=-1)
            id_mask = Y!=0
            id_true += ((predictions == Y)*id_mask).sum()
            id_total += id_mask.sum() 
            
            val_loss = loss(output_test.reshape(-1, 23), Y.reshape(-1))

            total_val_loss += val_loss.item()

    avg_val_loss = total_val_loss / len(test_loader)

    if epoch%5==0:
        print(f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Validation Loss: {avg_val_loss:.4f} |"
            f"Validation %id: {id_true/id_total}")


Epoch 1/21 | Train Loss: 2.3800 | Validation Loss: 2.1520 |Validation %id: 0.3512551486492157
Epoch 6/21 | Train Loss: 1.6591 | Validation Loss: 1.8471 |Validation %id: 0.4417881965637207
Epoch 11/21 | Train Loss: 1.5212 | Validation Loss: 1.7654 |Validation %id: 0.4660671651363373
Epoch 16/21 | Train Loss: 1.4399 | Validation Loss: 1.7183 |Validation %id: 0.48079729080200195
Epoch 21/21 | Train Loss: 1.3817 | Validation Loss: 1.6913 |Validation %id: 0.49071887135505676


In [35]:
with torch.no_grad():
        for (batch,) in test_loader:
            batch = batch.to(device)

            X = batch[:, :-1]
            Y = batch[:, 1:]

            output_test = v4(X)
            
            break
for out,y in zip(output_test,Y):
    print(f'Predicted vs target: \n{train.decode(out.argmax(dim=1))}\n{train.decode(y)}')

Predicted vs target: 
MTKSDLIAKIAEKTGIDKADATATLEAFMKVVKDSLKKGEKVYLRGFGSFIVKKRAEKIGRNISKNTTIDIPAKFIPSFKPGKAFVEKVKSVKVKKKKKKKKKKAKKKEE
MTKAEVIVAIADKTGVDKASVSTTLESFFEIVKDTLAQGDAVYVRGFGSFVNKHRAAKKARNISAKTTVMVPAHSVPGFKPAKEFTEQVKASVK
Predicted vs target: 
MTKVRVVVRIRVQGVGFRYFTRRQARQLGLAGWVRNLPDGSVEVVAEGSDEAVEAFLDALREGRGRAARSDAVEEEEEPEEGLRSDFEIRVGA
MERVTAVARGRVQGVGYRYYVSECAHATGIHGWVKNEPDGTVRIVAESSPAALDDFIRLAHARNDPVIRVDEITVEKGQATGEFHGFRVSW
Predicted vs target: 
MTARERTVLITNKLGLHARAAIKLVELAQQFDVEITVSQEEKQVPGGSGIMGLLMLESQQGHLVVVSAEGEDAEAALAAITELIAAGFDEEE
MPQSSRTVLIQNRLGLHARAAVKLVELAQSFEAVLTIQNQEGKEATADSVMGLLMLESAQGQNVTISAEGSQAEQALDAVCHLIEVKFDEDE
Predicted vs target: 
MTKLNIIITGKVQGVFFRATTKEKADQLGVTGFVKNEDDGSVEIVAQGEEEQLEEFIEWCKKGPPGAKVEKVEIKWGEEKKEFKNFSIKRK
MKHLNLKIHGKVQGVFFREKTKQLAQELGLNGYVKNTSDNCVEVVVEGDELALKKLLDWCKIGTEYAEVGEVEEKWSDEVGEFEEFKIVY
Predicted vs target: 
MTKTTVTHVTVSGRVQGVFFRAFTQEQARRLGVTGWVRNLPDGRVEAVFEGPAEAVEALVEWCRPGPPVARVDSVEVEWEEPTGEFFFEIRYVVVVVVVVHVVV
MTDRTRAHVYVSGKVQGVYFRATT